# Double ML IRM with sensitivity analysis (Python)

Two stages, so the computationally inexpensive propensity screening runs before the computationally intensive double ML model.

**Stage 1, propensity-score overlap screen.** For every encoded recommendation
except 13 and 14, fit only the treatment model `m(X) = P(A=1 | X)` per
observability tier and read the overlap of the cross-fitted propensity scores. A
recommendation is carried into the causal analysis only if, at the full tier, at
most 10% of its propensity scores fall outside `[0.05, 0.95]` and both treatment
arms exceed 100 patients. This is outcome-independent and fast.

**Stage 2, interactive regression model.** Only for the recommendations that
clear the screen, fit the IRM per outcome and tier, and add the native DoubleML sensitivity analysis the R package lacks: the robustness values (rv, rva) and their benchmarking against
observed confounders.

In [1]:
import os

if os.path.basename(os.getcwd()) == "Code":
    os.chdir("..")
print("cwd:", os.getcwd())

cwd: /Users/johannakastenberger/Library/Mobile Documents/com~apple~CloudDocs/Master Arbeit/master-thesis


In [2]:
## Imports and configuration
import numpy as np
import time
import pandas as pd

from doubleml import DoubleMLData, DoubleMLIRM
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

RNG = 123
np.random.seed(RNG)

## exported by R
DATA_PATH = "Data/dml_analysis_frame.parquet"
OUT_PATH = "Outputs/dml_irm_results_py.csv"

/opt/anaconda3/envs/thesis_dml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Column definitions
## confounders: pre-treatment common causes of adherence and outcome
W = [
    "anchor_age",
    "gender_male",
    "charlson_comorbidity_index",
    "sofa_score",
    "sapsii",
    "creatinine",
    "bun",
    "wbc",
    "platelets",
    "hemoglobin",
    "bilirubin",
    "albumin",
    "inr",
    "lactate",
    "ph_blood",
    "bicarbonate",
    "aki_baseline_stage",
]

## every encoded recommendation except 13 and 14 is a candidate for the causal
## analysis, at the treatment definition it would use: the _causal variant where a
## subgroup restriction applies, the plain adherence indicator otherwise. 13 and 14
## are excluded a priori, since a mean arterial pressure target is titrated to the
## patient's state and so is not an assignable action. rec 53 and 55 enter jointly
## via the combined measure. Inclusion is decided by the propensity-overlap screen
## below, not by adherence rate.
candidate_cols = [
    "rec_7_culture_before_antimicrobial_causal",
    "rec_8_lactate_measured_within_1h_causal",
    "rec_10_crystalloid_within_3h_causal",
    "rec_15_icu_6h_causal",
    "rec_16_antimicrobial_within_1h_causal",
    "rec_17_antimicrobial_within_1h_causal",
    "rec_42_bp_monitoring",
    "rec_43_crystalloid_first_line",
    "rec_44_balanced_over_saline_bolus_causal",
    "rec_45_crystalloid_no_albumin",
    "rec_53_55_norepi_first_line_causal",
    "rec_56_vasopressin_after_escalating_norepi_causal",
    "rec_57_epi_after_escalating_norepi_vaso",
    "rec_67_hfnc_over_conventional",
    "rec_68_first_oxygen_hfnc_causal",
    "rec_69_hfnc_over_alternating",
    "rec_71_low_tidal_volume",
    "rec_73_plateau_pressure_limited",
    "rec_79_iv_corticosteroids",
    "rec_89_active_fluid_removal_7d_causal",
    "rec_90_restrictive_transfusion_causal",
    "rec_91_early_enteral_nutrition_causal",
    "rec_92_insulin_started_at_glucose_180_causal",
    "rec_96_sodium_bicarb_causal",
    "rec_98_lmwh_ever",
]

## decision-point confounders added to the baseline W only for the model of the
## recommendation they belong to. The baseline W is anchored at sepsis onset and
## does not capture the patient's state at a later treatment decision, so for the
## recommendations whose decision occurs well after onset these columns carry the
## key decision-driving variables, measured before that decision.
extra_covariates = {
    "rec_56_vasopressin_after_escalating_norepi_causal": [
        "cf_rec56_norepi_rate_escalation",
        "cf_rec56_lactate",
        "cf_rec56_map",
    ],
    "rec_89_active_fluid_removal_7d_causal": [
        "cf_rec89_peak_vasopressor",
        "cf_rec89_cum_fluid_bolus",
        "cf_rec89_hours_to_stabilization",
    ],
    "rec_90_restrictive_transfusion_causal": [
        "cf_rec90_lactate",
        "cf_rec90_map",
        "cf_rec90_cardiac",
    ],
    "rec_91_early_enteral_nutrition_causal": [
        "cf_rec91_early_shock",
    ],
    "rec_92_insulin_started_at_glucose_180_causal": [
        "cf_rec92_qualifying_glucose",
    ],
    "rec_96_sodium_bicarb_causal": [
        "cf_rec96_ph",
        "cf_rec96_aki_stage",
        "cf_rec96_lactate",
        "cf_rec96_vaso_burden",
    ],
}

outcomes = ["mort_28d", "icu_mortality", "aki_new"]

tiers = ["full", "well_observed", "well_observed_strict"]

## confounders to benchmark the robustness value against: an unmeasured
## confounder "as strong as" one of these is the interpretable reference
benchmark_vars = ["sofa_score", "sapsii"]

## pre-specified inclusion gate for the causal analysis, applied at the full tier:
## a recommendation is carried in only if at most MAX_FRAC_OUTSIDE of its propensity
## scores fall outside [OVERLAP_LO, OVERLAP_HI] (positivity, read as a share of the
## sample rather than a single percentile), and both treatment arms exceed MIN_ARM
## so the estimate is not driven by a handful of patients
OVERLAP_LO, OVERLAP_HI = 0.05, 0.95
MAX_FRAC_OUTSIDE = 0.10
MIN_ARM = 100

In [4]:
## Load the analysis frame exported from R
df = pd.read_parquet("Outputs/dml_analysis_frame.parquet")
print(df.shape)
print(df.columns.tolist())

(21189, 65)
['subject_id', 'hadm_id', 'stay_id', 'rec_7_culture_before_antimicrobial_causal', 'rec_8_lactate_measured_within_1h_causal', 'rec_10_crystalloid_within_3h_causal', 'rec_15_icu_6h_causal', 'rec_16_antimicrobial_within_1h_causal', 'rec_17_antimicrobial_within_1h_causal', 'rec_42_bp_monitoring', 'rec_43_crystalloid_first_line', 'rec_44_balanced_over_saline_bolus_causal', 'rec_45_crystalloid_no_albumin', 'rec_53_55_norepi_first_line_causal', 'rec_56_vasopressin_after_escalating_norepi_causal', 'rec_57_epi_after_escalating_norepi_vaso', 'rec_67_hfnc_over_conventional', 'rec_68_first_oxygen_hfnc_causal', 'rec_69_hfnc_over_alternating', 'rec_71_low_tidal_volume', 'rec_73_plateau_pressure_limited', 'rec_79_iv_corticosteroids', 'rec_89_active_fluid_removal_7d_causal', 'rec_90_restrictive_transfusion_causal', 'rec_91_early_enteral_nutrition_causal', 'rec_92_insulin_started_at_glucose_180_causal', 'rec_96_sodium_bicarb_causal', 'rec_98_lmwh_ever', 'cf_rec56_norepi_rate_escalation', 'c

In [5]:
## One-model wrapper
def run_dml(treat_col, outcome_col, tier, do_benchmark=False):

    ## baseline confounders plus any decision-point confounders for this rec
    W_rec = W + extra_covariates.get(treat_col, [])

    ## result template so every row has the same columns, NA where not computed
    out = {
        "treat_col": treat_col,
        "outcome": outcome_col,
        "tier": tier,
        "n": np.nan,
        "n_treated": np.nan,
        "p0": np.nan,
        "ate": np.nan,
        "se": np.nan,
        "ci_lo": np.nan,
        "ci_hi": np.nan,
        "pval": np.nan,
        "ps_p01": np.nan,
        "ps_p99": np.nan,
        "rv": np.nan,
        "rva": np.nan,
    }
    for bv in benchmark_vars:
        out[f"cf_y_{bv}"] = np.nan
        out[f"cf_d_{bv}"] = np.nan

    ## assemble analysis frame: eligible stays only (A not missing)
    keep = ["subject_id", "hadm_id", "stay_id", treat_col, outcome_col,
            "well_observed", "well_observed_strict"] + W_rec
    d = df[keep].copy()
    d = d.rename(columns={treat_col: "A", outcome_col: "Y"})
    d = d[d["A"].notna()]

    # observability tier
    if tier == "well_observed":
        d = d[d["well_observed"] == True]
    elif tier == "well_observed_strict":
        d = d[d["well_observed_strict"] == True]

    d = d[d["Y"].notna()]
    d["A"] = d["A"].astype(int)
    d["Y"] = d["Y"].astype(int)

    ## guard: too few rows or no treatment variation
    if len(d) < 100 or d["A"].nunique() < 2:
        out["n"] = len(d)
        return out

    ## median imputation plus a missingness indicator
    X_cols = []
    for col in W_rec:
        miss = d[col].isna()
        if miss.any():
            flag = col + "_miss"
            d[flag] = miss.astype(int)
            d.loc[miss, col] = d[col].median()
            X_cols.append(flag)
        X_cols.append(col)

    d = d.dropna(subset=["Y", "A"] + X_cols)

    ## DoubleML data container
    dml_data = DoubleMLData(d, y_col="Y", d_cols="A", x_cols=X_cols)

    ## random-forest nuisance learners (parallelised over all cores)
    ml_g = RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                                 random_state=RNG, n_jobs=-1)
    ml_m = RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                  random_state=RNG, n_jobs=-1)

    ## IRM with 5-fold cross-fitting repeated 5 times, propensity trimmed to [0.01, 0.99]
    dml = DoubleMLIRM(dml_data, ml_g=ml_g, ml_m=ml_m,
                      n_folds=5, n_rep=5, trimming_threshold=0.01)
    dml.fit(store_predictions=True)

    ci = dml.confint()

    ## propensity scores pooled across repetitions for the overlap diagnostic
    ps = np.asarray(dml.predictions["ml_m"]).ravel()
    ps = ps[~np.isnan(ps)]

    out.update({
        "n": int(d.shape[0]),
        "n_treated": int(d["A"].sum()),
        "p0": float(d.loc[d["A"] == 0, "Y"].mean()),
        "ate": float(np.ravel(dml.coef)[0]),
        "se": float(np.ravel(dml.se)[0]),
        "ci_lo": float(ci.iloc[0, 0]),
        "ci_hi": float(ci.iloc[0, 1]),
        "pval": float(np.ravel(dml.pval)[0]),
        "ps_p01": float(np.quantile(ps, 0.01)),
        "ps_p99": float(np.quantile(ps, 0.99)),
    })

    ## sensitivity: robustness values always (cheap)
    ## benchmarking only on request
    try:
        dml.sensitivity_analysis()
        out["rv"] = float(np.ravel(dml.sensitivity_params["rv"])[0])
        out["rva"] = float(np.ravel(dml.sensitivity_params["rva"])[0])

        if do_benchmark:
            for bv in benchmark_vars:
                bset = [bv] + ([bv + "_miss"] if (bv + "_miss") in X_cols else [])
                bset = [c for c in bset if c in X_cols]
                if bset:
                    b = dml.sensitivity_benchmark(benchmarking_set=bset)
                    out[f"cf_y_{bv}"] = float(b["cf_y"].iloc[0])
                    out[f"cf_d_{bv}"] = float(b["cf_d"].iloc[0])
    except Exception as e:
        print(f"  sensitivity failed [{treat_col} | {outcome_col} | {tier}]: {e}")

    return out

In [6]:
import time
t0 = time.time()
r = run_dml("rec_16_antimicrobial_within_1h_causal", "mort_28d", "full")
print(round(time.time() - t0, 1), "s"); r

34.2 s


{'treat_col': 'rec_16_antimicrobial_within_1h_causal',
 'outcome': 'mort_28d',
 'tier': 'full',
 'n': 5069,
 'n_treated': 1050,
 'p0': 0.29136601144563323,
 'ate': -0.049291587818469026,
 'se': 0.0169498402942986,
 'ci_lo': -0.08251266433900006,
 'ci_hi': -0.016070511297937982,
 'pval': 0.0036364869118388626,
 'ps_p01': 0.024612218334998544,
 'ps_p99': 0.5925165191488719,
 'rv': 0.04089921787624069,
 'rva': 0.01794669764650298,
 'cf_y_sofa_score': nan,
 'cf_d_sofa_score': nan,
 'cf_y_sapsii': nan,
 'cf_d_sapsii': nan}

In [13]:
# Propensity-only wrapper: fits just the treatment model m(X) = P(A=1 | X)
## This is the overlap screen. It uses the same learner and features the IRM would
## use for its propensity nuisance, but skips the outcome model, the effect
## estimate and the sensitivity analysis, so it is outcome-independent and fast.
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def run_propensity(treat_col, tier, n_folds=5, n_rep=1):

    # baseline confounders plus any decision-point confounders for this rec,
    # matching the covariate set the IRM uses for its propensity nuisance
    W_rec = W + extra_covariates.get(treat_col, [])

    out = {
        "treat_col": treat_col,
        "tier": tier,
        "n": np.nan,
        "n_treated": np.nan,
        "prevalence": np.nan,
        "ps_p01": np.nan,
        "ps_p05": np.nan,
        "ps_p50": np.nan,
        "ps_p95": np.nan,
        "ps_p99": np.nan,
        "frac_below_05": np.nan,
        "frac_above_95": np.nan,
    }

    ## eligible stays only (A not missing), then the observability tier
    keep = ["subject_id", "hadm_id", "stay_id", treat_col,
            "well_observed", "well_observed_strict"] + W_rec
    d = df[keep].copy()
    d = d.rename(columns={treat_col: "A"})
    d = d[d["A"].notna()]

    if tier == "well_observed":
        d = d[d["well_observed"] == True]
    elif tier == "well_observed_strict":
        d = d[d["well_observed_strict"] == True]

    d["A"] = d["A"].astype(int)

    ## same feature construction as run_dml: median imputation plus missingness flag
    X_cols = []
    for col in W_rec:
        miss = d[col].isna()
        if miss.any():
            flag = col + "_miss"
            d[flag] = miss.astype(int)
            d.loc[miss, col] = d[col].median()
            X_cols.append(flag)
        X_cols.append(col)

    d = d.dropna(subset=["A"] + X_cols)

    ## guard: too few rows or no treatment variation
    if len(d) < 100 or d["A"].nunique() < 2:
        out["n"] = len(d)
        out["n_treated"] = int(d["A"].sum())
        return out

    X = d[X_cols].to_numpy()
    A = d["A"].to_numpy()

    ## cross-fitted out-of-fold propensity, averaged over repetitions, so the
    ## scores are honest (never predicting a stay from a model trained on it)
    ps_acc = np.zeros(len(d))
    for r in range(n_rep):
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True,
                              random_state=RNG + r)
        clf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                     random_state=RNG + r, n_jobs=-1)
        ps_acc += cross_val_predict(clf, X, A, cv=skf,
                                    method="predict_proba")[:, 1]
    ps = ps_acc / n_rep

    out.update({
        "n": int(len(d)),
        "n_treated": int(A.sum()),
        "prevalence": float(A.mean()),
        "ps_p01": float(np.quantile(ps, 0.01)),
        "ps_p05": float(np.quantile(ps, 0.05)),
        "ps_p50": float(np.quantile(ps, 0.50)),
        "ps_p95": float(np.quantile(ps, 0.95)),
        "ps_p99": float(np.quantile(ps, 0.99)),
        "frac_below_05": float((ps < 0.05).mean()),
        "frac_above_95": float((ps > 0.95).mean()),
    })
    return out

In [14]:
# Propensity-score overlap screen (fast, outcome-independent)
## One treatment-model fit per candidate x tier, 25 x 3 = 75 fits, minutes not hours.
from tqdm import tqdm

ps_combos = [(t, tier) for t in candidate_cols for tier in tiers]

ps_rows = []
for t, tier in tqdm(ps_combos, desc="propensity screen", unit="fit"):
    try:
        ps_rows.append(run_propensity(t, tier))
    except Exception as e:
        print(f"ERROR {t} | {tier}: {e}")
        ps_rows.append({"treat_col": t, "tier": tier})

ps_df = pd.DataFrame(ps_rows)

PS_OUT_PATH = "Outputs/propensity_overlap_py.csv"
os.makedirs(os.path.dirname(PS_OUT_PATH), exist_ok=True)
ps_df.to_csv(PS_OUT_PATH, index=False)
print(f"wrote {PS_OUT_PATH}  ({ps_df.shape[0]} rows)")

## pre-specified inclusion gate, applied at the full tier: the share of propensity
## scores outside [OVERLAP_LO, OVERLAP_HI] must be at most MAX_FRAC_OUTSIDE, and the
## smaller treatment arm must exceed MIN_ARM. Reading overlap as a share of the
## sample rather than a single percentile avoids knife-edge calls where one tail
## just crosses the bound while the bulk overlaps.
ps_df["frac_outside"] = ps_df["frac_below_05"] + ps_df["frac_above_95"]
ps_df["n_control"] = ps_df["n"] - ps_df["n_treated"]
ps_df["min_arm"] = ps_df[["n_treated", "n_control"]].min(axis=1)
ps_df["adequate_overlap"] = ((ps_df["frac_outside"] <= MAX_FRAC_OUTSIDE) &
                             (ps_df["min_arm"] >= MIN_ARM))

included = [t for t in candidate_cols
           if bool(ps_df.loc[(ps_df.treat_col == t) & (ps_df.tier == "full"),
                             "adequate_overlap"].fillna(False).iloc[0])]
excluded = [t for t in candidate_cols if t not in included]

print(f"\n{len(included)} included, {len(excluded)} excluded by the overlap gate")
print("included:", included)
print("excluded:", excluded)

ps_df[ps_df.tier == "full"][
    ["treat_col", "n", "n_treated", "min_arm", "prevalence",
     "ps_p05", "ps_p95", "frac_below_05", "frac_above_95",
     "frac_outside", "adequate_overlap"]
]

propensity screen: 100%|██████████| 75/75 [02:44<00:00,  2.19s/fit]

wrote Outputs/propensity_overlap_py.csv  (75 rows)

15 included, 10 excluded by the overlap gate
included: ['rec_7_culture_before_antimicrobial_causal', 'rec_8_lactate_measured_within_1h_causal', 'rec_10_crystalloid_within_3h_causal', 'rec_15_icu_6h_causal', 'rec_17_antimicrobial_within_1h_causal', 'rec_44_balanced_over_saline_bolus_causal', 'rec_53_55_norepi_first_line_causal', 'rec_56_vasopressin_after_escalating_norepi_causal', 'rec_68_first_oxygen_hfnc_causal', 'rec_69_hfnc_over_alternating', 'rec_89_active_fluid_removal_7d_causal', 'rec_90_restrictive_transfusion_causal', 'rec_91_early_enteral_nutrition_causal', 'rec_92_insulin_started_at_glucose_180_causal', 'rec_96_sodium_bicarb_causal']
excluded: ['rec_16_antimicrobial_within_1h_causal', 'rec_42_bp_monitoring', 'rec_43_crystalloid_first_line', 'rec_45_crystalloid_no_albumin', 'rec_57_epi_after_escalating_norepi_vaso', 'rec_67_hfnc_over_conventional', 'rec_71_low_tidal_volume', 'rec_73_plateau_pressure_limited', 'rec_79_iv_corti

,treat_col,n,n_treated,min_arm,prevalence,ps_p05,ps_p95,frac_below_05,frac_above_95,frac_outside,adequate_overlap
0,rec_7_culture_before_antimicrobial_causal,18038,10620,7418,0.588757,0.163558,0.925488,0.005100,0.011143,0.016243,True
3,rec_8_lactate_measured_within_1h_causal,21189,11023,10166,0.520223,0.200270,0.868756,0.002926,0.000802,0.003728,True
6,rec_10_crystalloid_within_3h_causal,6476,801,801,0.123687,0.041356,0.262729,0.091260,0.000000,0.091260,True
9,rec_15_icu_6h_causal,11744,7127,4617,0.606863,0.312438,0.822384,0.001362,0.000511,0.001873,True
12,rec_16_antimicrobial_within_1h_causal,5069,1050,1050,0.207141,0.036850,0.517469,0.121918,0.000000,0.121918,False
15,rec_17_antimicrobial_within_1h_causal,12969,2514,2514,0.193847,0.049174,0.473953,0.053435,0.000000,0.053435,True
18,rec_42_bp_monitoring,6487,6393,94,0.985509,0.947668,0.999040,0.000000,0.944813,0.944813,False
21,rec_43_crystalloid_first_line,7262,7160,102,0.985954,0.947015,0.999667,0.000000,0.942440,0.942440,False
24,rec_44_balanced_over_saline_bolus_causal,3270,2150,1120,0.657492,0.251172,0.954262,0.000000,0.064526,0.064526,True
27,rec_45_crystalloid_no_albumin,7751,7418,333,0.957038,0.867652,0.997760,0.000000,0.653980,0.653980,False


In [10]:
# Full IRM grid, only for the recommendations that cleared the overlap gate
from tqdm import tqdm

combos = [(t, o, tier) for t in included for o in outcomes for tier in tiers]
print(f"{len(included)} recommendations included -> {len(combos)} fits")

results = []
for t, o, tier in tqdm(combos, desc="DML grid", unit="fit"):
    try:
        results.append(run_dml(t, o, tier))          # do_benchmark=False by default
    except Exception as e:
        print(f"ERROR {t} | {o} | {tier}: {e}")
        results.append({"treat_col": t, "outcome": o, "tier": tier})

results_df = pd.DataFrame(results)

15 recommendations included -> 135 fits


DML grid: 100%|██████████| 135/135 [1:41:38<00:00, 45.17s/fit]


In [11]:
## Save
import os

OUT_PATH = "Outputs/dml_irm_results_py.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
results_df.to_csv(OUT_PATH, index=False)
print(f"wrote {OUT_PATH}  ({results_df.shape[0]} rows)")
results_df.head()

wrote Outputs/dml_irm_results_py.csv  (135 rows)


,treat_col,outcome,tier,n,n_treated,p0,ate,se,ci_lo,ci_hi,pval,ps_p01,ps_p99,rv,rva,cf_y_sofa_score,cf_d_sofa_score,cf_y_sapsii,cf_d_sapsii
0,rec_7_culture_before_antimicrobial_causal,mort_28d,full,18038,10620.0,0.153276,0.051145,0.006984,0.037432,0.064833,2.417763e-13,0.065801,0.951349,0.048953,0.038187,NaN,NaN,NaN,NaN
1,rec_7_culture_before_antimicrobial_causal,mort_28d,well_observed,13685,8388.0,0.174627,0.040225,0.008433,0.023696,0.056754,1.843320e-06,0.072389,0.954439,0.036370,0.023980,NaN,NaN,NaN,NaN
2,rec_7_culture_before_antimicrobial_causal,mort_28d,well_observed_strict,6978,2718.0,0.159155,0.066434,0.009929,0.047010,0.085895,2.221136e-11,0.030109,0.820054,0.064237,0.048710,NaN,NaN,NaN,NaN
3,rec_7_culture_before_antimicrobial_causal,icu_mortality,full,18038,10620.0,0.090995,0.020041,0.005898,0.008497,0.031600,6.783992e-04,0.065056,0.951132,0.023491,0.012204,NaN,NaN,NaN,NaN
4,rec_7_culture_before_antimicrobial_causal,icu_mortality,well_observed,13685,8388.0,0.103266,0.012641,0.007394,-0.001780,0.027134,8.733218e-02,0.071677,0.954150,0.014168,0.000589,NaN,NaN,NaN,NaN


In [14]:
from tqdm.auto import tqdm

res = pd.read_csv("Outputs/dml_irm_results_py.csv")

## full-tier models with a significant effect, benchmarked against the strongest
## measured confounders
sel = res[(res.tier == "full") & res.ate.notna() & (res.pval < 0.01)]
final_models = list(zip(sel.treat_col, sel.outcome, sel.tier))
print(f"{len(final_models)} models to benchmark")

bench_rows = []
for (t, o, tier) in tqdm(final_models, desc="benchmark", unit="model"):
    try:
        bench_rows.append(run_dml(t, o, tier, do_benchmark=True))
    except Exception as e:
        print(f"ERROR {t} | {o} | {tier}: {e}")
        bench_rows.append({"treat_col": t, "outcome": o, "tier": tier})

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv("Outputs/dml_irm_benchmark_py.csv", index=False)
print(bench_df[["treat_col", "outcome", "ate", "rv",
                "cf_d_sofa_score", "cf_d_sapsii"]].to_string(index=False))

21 models to benchmark


benchmark: 100%|██████████| 21/21 [1:05:28<00:00, 187.05s/model]

                                   treat_col       outcome       ate       rv  cf_d_sofa_score  cf_d_sapsii
   rec_7_culture_before_antimicrobial_causal      mort_28d  0.050481 0.048288         0.000000     0.000000
   rec_7_culture_before_antimicrobial_causal icu_mortality  0.018813 0.022093         0.000000     0.004195
                        rec_15_icu_6h_causal      mort_28d  0.043077 0.048640         0.000290     0.000000
                        rec_15_icu_6h_causal icu_mortality  0.023652 0.033287         0.003122     0.002073
                        rec_15_icu_6h_causal       aki_new -0.058721 0.055609         0.000000     0.000000
    rec_44_balanced_over_saline_bolus_causal      mort_28d -0.067037 0.083369         0.000000     0.014875
    rec_44_balanced_over_saline_bolus_causal icu_mortality -0.051951 0.073079         0.026118     0.000000
          rec_53_55_norepi_first_line_causal      mort_28d  0.058606 0.047912         0.346802     0.113189
          rec_53_55_norepi_f